# 🔄 Auto Loader — Incremental File Ingestion & Schema Evolution

This notebook demonstrates **Databricks Auto Loader** (`cloudFiles` source) for incrementally processing CSV files as they arrive in cloud storage. It covers **checkpointing for exactly-once processing**, **schema inference with hints**, **schema evolution modes** (`addNewColumns` vs `rescue`), and **file provenance tracking** via `_metadata.file_name`.

> 📖 **Each step below has its own markdown explanation cell — what we're doing, why, and key insights.**

## 📁 Step 1: Create Input Directories

**What:** Creating 10 date-partitioned directories under the autoloader input volume.

**Why:** Auto Loader needs a landing zone to monitor. Nested `year/month/day` folders simulate a real data lake layout.

**💡 Insight:** The `/*/` wildcard in the load path (Step 4) will recursively scan all these subdirectories — this is how Auto Loader handles partitioned folder structures.

In [0]:
# Create autoloader input location in volumne
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/01')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/02')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/03')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/04')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/05')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/06')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/07')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/08')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/09')
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/autoloader_input/2010/12/10')

True

## 🔒 Step 2: Create Checkpoint Location

**What:** Creating a checkpoint directory in the volume.

**Why:** The checkpoint is the backbone of **exactly-once processing**. It tracks which files have already been ingested — without it, every run reprocesses all files.

**💡 Insight:** Checkpoint tracks file-level progress; schema location tracks the inferred schema — they are distinct concepts.

In [0]:
# Create checkpoint loaction in volumne
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint')

True

## 📄 Step 3: Copy Initial CSV Files

**What:** Copying 3 CSV files (Dec 1, 2, 3) from Databricks sample datasets into the input folders.

**Why:** Provides the initial batch of data for the first Auto Loader run.

In [0]:
# Copy files t nested locations
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv","/Volumes/dev/bronze/landing/autoloader_input/2010/12/01")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv","/Volumes/dev/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv","/Volumes/dev/bronze/landing/autoloader_input/2010/12/03")

True

## ⚙️ Step 4: Define Auto Loader Stream

**What:** Configuring the streaming read using Auto Loader (`cloudFiles` source).

| Option | Purpose |
| --- | --- |
| `cloudFiles.format = csv` | File format to watch for |
| `pathGlobFilter = *.csv` | Only pick up CSV files |
| `cloudFiles.schemaHints` | Force `Quantity` as int, `UnitPrice` as double |
| `cloudFiles.schemaLocation` | Where schema is inferred and tracked |
| `load path = .../*/` | Recursively scan all date subdirectories |

**Why:** Schema hints are critical — without them, `Quantity` and `UnitPrice` could be inferred as `string`, breaking downstream aggregations.

**💡 Insight:** The `/*/` wildcard enables recursive directory scanning across all date partitions.

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","*.csv")
    .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
    .option("cloudFiles.schemaLocation","/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint/1/")
    .option("Header","true")
    .load("/Volumes/dev/bronze/landing/autoloader_input/*/")
)

## 🚀 Step 5: First Stream Execution

**What:** Writing the stream to Delta table `dev.bronze.invoice_al_1` with:
* `__file` column from `_metadata.file_name` for file provenance
* `outputMode("append")` — only insert new rows
* `trigger(availableNow=True)` — process all available files, then stop

**Why:** `availableNow=True` makes streaming behave like batch — process everything available, then terminate. Ideal for scheduled incremental loads.

**💡 Insight:** `_metadata.file_name` is a built-in column Auto Loader provides automatically — no extra config needed.

In [0]:
from pyspark.sql.functions import col
(
    df.withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint/1/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("dev.bronze.invoice_al_1")
)

## 🔎 Step 6: Verify Loaded Data

**What:** Querying 20 rows from `dev.bronze.invoice_al_1`.

**Why:** Confirms schema hints worked — `Quantity` is `int`, `UnitPrice` is `double`, not strings.

In [0]:
%sql
select * from dev.bronze.invoice_al_1 limit 20;

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_rescued_data,__file
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,null,2010-12-01.csv
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,null,2010-12-01.csv
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,null,2010-12-01.csv
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,null,2010-12-01.csv
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,null,2010-12-01.csv
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,null,2010-12-01.csv
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,null,2010-12-01.csv
536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,null,2010-12-01.csv
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,null,2010-12-01.csv
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,null,2010-12-01.csv


## 📊 Step 7: Count Rows Per File

**What:** Counting rows grouped by file name.

**Why:** Verifies that only the 3 files present were loaded — no more, no less.

**Expected:** `2010-12-01.csv` → 3,108 | `2010-12-02.csv` → 2,109 | `2010-12-03.csv` → 2,202

In [0]:
%sql
select __file,count(__file) as total_counts from dev.bronze.invoice_al_1
group by __file

__file,total_counts
2010-12-01.csv,3108
2010-12-02.csv,2109
2010-12-03.csv,2202


## ➕ Step 8: Add a New File

**What:** Copying `2010-12-05.csv` into the input directory.

**Why:** Tests the core promise of Auto Loader — incremental processing. A new file arrives; Auto Loader should pick up ONLY the new file on the next run.

In [0]:
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-05.csv","/Volumes/dev/bronze/landing/autoloader_input/2010/12/05")

True

## 🔄 Step 9: Re-run the Same Stream

**What:** Re-running the exact same streaming write with the same checkpoint.

**Why:** Same code + same checkpoint = Auto Loader knows it already processed Dec 1–3 and will only ingest the new Dec 5 file.

**💡 Insight:** This proves exactly-once processing — no duplicates on re-run.

In [0]:
from pyspark.sql.functions import col
(
    df.withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint/1/")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("dev.bronze.invoice_al_1")
)

## 📊 Step 10: Verify Incremental Load

**What:** Counting rows per file after the incremental run.

**Why:** Confirms only the new file was added — old counts should be unchanged.

**Expected:** Dec 1–3 unchanged (3,108 / 2,109 / 2,202) + new Dec 5 (2,725).

In [0]:
%sql
select __file,count(__file) as total_counts from dev.bronze.invoice_al_1
group by __file

__file,total_counts
2010-12-01.csv,3108
2010-12-02.csv,2109
2010-12-03.csv,2202
2010-12-05.csv,2725


## 🧬 Step 11: Schema Evolution with mergeSchema

**What:** Re-running the stream with `.option("mergeSchema", "true")`.

**Why:** Tests what happens when new columns appear in incoming data. Auto Loader's default mode (`addNewColumns`) adds new columns to the table schema automatically.

**💡 Insight:** Schema evolved from 10 to 13 columns — `Tax`, `ShippingMethod`, `Notes` were auto-added.

In [0]:
# hecking default mode of schema evolution
from pyspark.sql.functions import col

(
    df.withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint/1/")
    .outputMode("append")
    .option("mergeSchema","true")
    .trigger(availableNow=True)
    .toTable("dev.bronze.invoice_al_1")
)

## 📊 Step 12: Verify After Schema Evolution

**What:** Counting rows per file after the mergeSchema run.

**Why:** Confirms new files with new columns were loaded successfully.

In [0]:
%sql
select __file,count(__file) as total_counts from dev.bronze.invoice_al_1
group by __file

__file,total_counts
2010-12-01.csv,3108
2010-12-02.csv,2109
2010-12-03.csv,2202
2010-12-05.csv,2725
2010-12-06.csv,5


## 📊 Step 13: Final Count for Table 1

**What:** Final row count per file for `dev.bronze.invoice_al_1`.

**Why:** Shows all files loaded so far, including rescue test files (Dec 6) with intentionally bad data.

In [0]:
%sql
select __file,count(*) from dev.bronze.invoice_al_1
group by __file


__file,count(*)
2010-12-01.csv,3108
2010-12-02.csv,2109
2010-12-03.csv,2202
2010-12-05.csv,2725
2010-12-06.csv,5


## 🛡️ Step 14: Configure Rescue Mode

**What:** Creating a NEW Auto Loader stream with `cloudFiles.schemaEvolutionMode = "rescue"` and a fresh schema location (`checkpoint/2/`).

**Why:** Rescue mode captures data that doesn't fit the schema into `_rescued_data` as JSON — instead of adding new columns to the schema.

**💡 Key Insight:** This is the **production-safe** approach. Schema stays stable; unexpected data is quarantined — no broken downstream pipelines.

In [0]:
# RESCUE MODE
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","*.csv")
    .option("cloudFiles.schemaHints","Quantity int, UnitPrice double")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.schemaLocation","/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint/2/")
    .option("Header","true")
    .load("/Volumes/dev/bronze/landing/autoloader_input/*/")
)

## 🚀 Step 15: Write to New Table (Rescue Mode)

**What:** Writing the rescue-mode stream to a new table `dev.bronze.invoice_al_2` with a new checkpoint.

**Why:** Separate table and checkpoint to test rescue mode independently without polluting the first experiment.

In [0]:
#   
from pyspark.sql.functions import col

(
    df.withColumn("__file",col("_metadata.file_name"))
    .writeStream
    .option("checkpointLocation","/Volumes/dev/bronze/landing/checkpoint/autoloader_checkpoint/2/")
    .outputMode("append")
    .option("mergeSchema","true")
    .trigger(availableNow=True)
    .toTable("dev.bronze.invoice_al_2")
    )

## 📊 Step 16: Count Rows (Rescue Table)

**What:** Counting rows per file for the rescue-mode table.

**Why:** Shows all 7 files loaded including rescue test files (Dec 4, 6, 7) with intentionally bad data.

In [0]:
%sql
select __file,count(*) from dev.bronze.invoice_al_2
group by __file

__file,count(*)
2010-12-06.csv,5
2010-12-07.csv,5
2010-12-05.csv,2725
2010-12-01.csv,3108
2010-12-02.csv,2109
2010-12-03.csv,2202
2010-12-04.csv,5


## 🔍 Step 17: Inspect Rescued Data

**What:** Querying rows where `_rescued_data IS NOT NULL` — data that didn't fit the schema was quarantined here.

**Why:** This is the payoff of rescue mode — inspect and handle unexpected data deliberately.

**💡 Key Insight:** Rescue mode captures:
1. **Type mismatches** — `Quantity="abc"` couldn't cast to `int` → rescued, main column is `null`
2. **Unknown columns** — `InvoiceNumber` not in schema → rescued as JSON
3. **File path** — `_file_path` in rescued JSON gives full traceability

---

## 🎯 Key Takeaways for Interviews

1. **`addNewColumns` (default) is convenient but risky** — schema changes propagate downstream automatically, potentially breaking pipelines.
2. **Rescue mode is production-safe** — quarantines unexpected data in `_rescued_data` for deliberate handling.
3. **`availableNow=True` bridges streaming and batch** — process everything available, then stop. Perfect for scheduled loads.
4. **Checkpointing guarantees exactly-once** — re-running with the same checkpoint never creates duplicates.
5. **Schema hints prevent silent type issues** — without hints, `Quantity` could be inferred as `string`.

In [0]:
%sql
select * from dev.bronze.invoice_al_2 where _rescued_data is not null;

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Discount,Tax,ShippingMethod,Notes,_rescued_data,__file
null,79323L,RESCUE TEST ITEM ONE,null,2010-12-06 09:00:00,4.55,17850.0,null,10,null,null,null,"{""Quantity"":""abc"",""InvoiceNumber"":""536999"",""_file_path"":""/Volumes/dev/bronze/landing/autoloader_input/2010/12/04/2010-12-04.csv""}",2010-12-04.csv
null,79323P,RESCUE TEST ITEM TWO,3,2010-12-06 09:00:00,2.75,17850.0,null,5,null,null,null,"{""InvoiceNumber"":""536999"",""_file_path"":""/Volumes/dev/bronze/landing/autoloader_input/2010/12/04/2010-12-04.csv""}",2010-12-04.csv
null,84406B,RESCUE TEST ITEM THREE,5,2010-12-06 09:15:00,3.39,null,null,0,null,null,null,"{""InvoiceNumber"":""537000"",""_file_path"":""/Volumes/dev/bronze/landing/autoloader_input/2010/12/04/2010-12-04.csv""}",2010-12-04.csv
null,84029G,RESCUE TEST ITEM FOUR,2,2010-12-06 09:20:00,1.95,17850.0,null,15,null,null,null,"{""InvoiceNumber"":""537001"",""_file_path"":""/Volumes/dev/bronze/landing/autoloader_input/2010/12/04/2010-12-04.csv""}",2010-12-04.csv
null,84029E,RESCUE TEST ITEM FIVE,7,2010-12-06 09:25:00,5.5,17850.0,null,20,null,null,null,"{""InvoiceNumber"":""537002"",""_file_path"":""/Volumes/dev/bronze/landing/autoloader_input/2010/12/04/2010-12-04.csv""}",2010-12-04.csv
